In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pii_utils import PiiCrypto

# PII configuration
pii_key = dbutils.secrets.get(
    scope="pii_scope",
    key="pii_aes_key"
)

crypto = PiiCrypto(encryption_key=pii_key)

pii_columns = [
    "customer_first_name",
    "customer_last_name",
    "customer_email"
]


# Read Bronze tables as streams
yellow_bronze_df = (
    spark.readStream
    .table("bronze.yellow_trips_raw")
)

green_bronze_df = (
    spark.readStream
    .table("bronze.green_trips_raw")
)


# Decrypt PII before cleaning
yellow_decrypted_df = crypto.decrypt_columns(
    yellow_bronze_df,
    pii_columns
)

green_decrypted_df = crypto.decrypt_columns(
    green_bronze_df,
    pii_columns
)


In [0]:
def clean_trip_stream(
    df,
    pickup_column,
    dropoff_column,
    columns_to_drop=None
):
    valid_trip = (
        F.col(pickup_column).isNotNull() &
        F.col(dropoff_column).isNotNull() &
        (F.col(dropoff_column) > F.col(pickup_column)) &
        F.col("PULocationID").isNotNull() &
        F.col("DOLocationID").isNotNull() &
        (F.col("PULocationID") > 0) &
        (F.col("DOLocationID") > 0) &
        (F.col("trip_distance") >= 0)
    )

    cleaned_df = df.filter(valid_trip)

    if columns_to_drop:
        cleaned_df = cleaned_df.drop(*columns_to_drop)

    return cleaned_df


yellow_silver_df = clean_trip_stream(
    yellow_bronze_df,
    pickup_column="tpep_pickup_datetime",
    dropoff_column="tpep_dropoff_datetime"
)

green_silver_df = clean_trip_stream(
    green_bronze_df,
    pickup_column="lpep_pickup_datetime",
    dropoff_column="lpep_dropoff_datetime",
    columns_to_drop=["ehail_fee"]
)

In [0]:
# Encrypt PII before writing to Silver
yellow_silver_df = crypto.encrypt_columns(
    yellow_clean_df,
    pii_columns
)

green_silver_df = crypto.encrypt_columns(
    green_clean_df,
    pii_columns
)

storage_account_name = "stdevnortheuropebfn0"

yellow_checkpoint_path = (
    f"abfss://data@{storage_account_name}.dfs.core.windows.net/"
    "checkpoints/silver/yellow"
)

green_checkpoint_path = (
    f"abfss://data@{storage_account_name}.dfs.core.windows.net/"
    "checkpoints/silver/green"
)

In [0]:
# Start the Yellow Taxi stream
yellow_query = (
    yellow_silver_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        yellow_checkpoint_path
    )
    .partitionBy("year", "month", "day")
    .trigger(processingTime="20 seconds")
    .outputMode("append")
    .queryName("silver_yellow_taxi_processing")
    .table("silver.yellow_trips")
)

display(spark.readStream.table("silver.yellow_trips"))

In [0]:
# Start the Green Taxi stream
green_query = (
    green_silver_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        green_checkpoint_path
    )
    .partitionBy("year", "month", "day")
    .trigger(processingTime="20 seconds")
    .outputMode("append")
    .queryName("silver_green_taxi_processing")
    .table("silver.green_trips")
)

display(spark.readStream.table("silver.green_trips"))